# Análise individual: Declaração da OCDE

**Projeto:** *As Relações Brasil-China e a Política de Governança Brasileira sobre Inteligência Artificial* (FAPESP, Processo nº 2026/04597-1)
**Etapa:** 02 (análise individual), conforme a skill 02 ([`skill_análise_individual.md`](skill_análise_individual.md))
**Insumo:** `OCDE_extracao.json`, produzido pela skill 01 ([`skill_extração.md`](skill_extração.md))

---

## 1. Do que se trata

Este notebook analisa, de forma **individual**, o texto da Declaração da OCDE. Ele faz parte do fluxo de análise da pasta *Declarações IA*: a skill 01 extrai o conteúdo integral das declarações; esta etapa (skill 02) analisa cada declaração isoladamente; a skill 03 compara as declarações entre si.

## 2. Finalidade

A finalidade deste notebook é **gerar visualizações e interpretações sobre o documento analisado**, por meio de gráficos, tabelas e medidas textuais. As análises permitem conhecer:

- a **extensão** e a estrutura do texto;
- os **termos mais comuns** e como se distribuem ao longo do documento;
- as **expressões recorrentes**;
- o **enfoque temático** e as **proporções** entre temas;
- a **ênfase normativa** (o grau de compromisso expresso pelos verbos modais).

## 3. Insumo e recorte do conteúdo

São utilizados **apenas os resultados da skill 01**, ou seja, o arquivo JSON da declaração gerado por aquela etapa. Desse arquivo, interessam **somente**:

- o **título** do texto (`document.title`), usado para identificar o documento;
- o **conteúdo integral e literal de cada parágrafo ou item** (`sections[].text`), que é o único conteúdo analisado.

O identificador de cada unidade (`id`) é usado apenas como índice de posição no documento. **Todo conteúdo periférico não é utilizado**: subtítulos internos, numeração, categoria estrutural, página, lista de signatários e demais metadados ficam fora da análise. A função `carregar_extracao` descarta esse conteúdo no momento da leitura.

## 4. Método (visão geral)

| Etapa | Procedimento |
|---|---|
| Verificação | Integridade do JSON, origem na skill 01 e pendências de revisão humana |
| Pré-processamento | spaCy (`en_core_web_sm`): tokenização, reunião de compostos com hífen, lematização, classes gramaticais, *stopwords* |
| Frequências | Frequência absoluta, por 1.000 tokens, alcance (unidades em que o termo aparece) e dispersão (DP normalizado de Gries) |
| Expressões | Bigramas de tokens de conteúdo adjacentes no texto original |
| Enfoque temático | Codebook de categorias (`codebook_declaracoes.json`), com IC por bootstrap e teste de robustez |
| Ênfase normativa | Verbos modais por 1.000 tokens |

O detalhamento de cada procedimento está na configuração, nas funções comentadas e no texto que acompanha cada visualização.

## 5. Transparência das escolhas

**Todo método utilizado e todas as escolhas tomadas pelo pesquisador estão presentes e verificáveis** em dois lugares:

1. **no código**: os parâmetros ficam reunidos na célula de configuração, e cada função traz comentários sobre o que faz;
2. **na descrição que segue cada visualização**, composta por: *Método e escolhas*, *Como ler*, *Interpretação* e *Fragilidades e ajustes possíveis*.

As tabelas completas (e não apenas os recortes exibidos nos gráficos) são exportadas para conferência, junto com um registro de execução com parâmetros, versões das bibliotecas e *hash* dos insumos.

## 6. Princípios

Todo o conteúdo deste notebook segue os princípios da **skill 02** (análise individual das declarações). Por isso, **não incorpora análise conjunta**: nenhum resultado aqui é comparado com outro documento. A comparação entre declarações é feita em notebook próprio, segundo a skill 03.

> As interpretações são redigidas **somente após a execução** do notebook, com base nos valores efetivamente obtidos. Enquanto isso, os campos de interpretação ficam marcados como pendentes.

## 7. Configuração

Todos os parâmetros editáveis estão nesta célula. Qualquer alteração deve ser registrada (o registro de execução, ao final, grava os valores usados).

In [ ]:
import hashlib
import json
import math
import platform
import re
import unicodedata
import zlib
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
from IPython.display import Markdown, display
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MaxNLocator

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 140)

# =====================================================================
# CONFIGURAÇÃO — parâmetros editáveis pelo pesquisador
# =====================================================================
SIGLA = "OCDE"
ARQUIVO_JSON = "OCDE_extracao.json"            # entregável da skill 01
ARQUIVO_CODEBOOK = "codebook_declaracoes.json"     # codebook compartilhado da pasta

# Saídas (figuras, tabelas, registro). Resultados anteriores nunca são sobrescritos:
# para uma nova execução com mudanças, altere VERSAO_SAIDA (v2, v3, ...).
NOME_SAIDA = SIGLA
VERSAO_SAIDA = "v1"
SALVAR_SAIDAS = True
SOBRESCREVER = False

# Recorte do conteúdo
INCLUIR_TITULO_NA_CONTAGEM = False   # o título identifica o documento; por padrão não é contado

# Pré-processamento
IDIOMA_ESPERADO = "en"
MODELO_SPACY = "en_core_web_sm"
CLASSES_DE_CONTEUDO = {"NOUN", "PROPN", "VERB", "ADJ", "ADV"}
TAMANHO_MINIMO_LEMA = 2
STOPWORDS_ADICIONAIS = []    # lemas a remover além da lista do spaCy (decisão do pesquisador)
STOPWORDS_MANTIDAS = []      # lemas da lista do spaCy que devem ser mantidos (decisão do pesquisador)
CORRECOES_DE_LEMA = {
    "datum": "data",         # o spaCy lematiza "data" como "datum"; mantém-se "data", forma usada nos textos
}

# Recortes de visualização (as tabelas exportadas são sempre completas)
TOP_N_LEMAS = 25
TOP_N_DISPERSAO = 15
TOP_N_BIGRAMAS = 20
FREQ_MINIMA_BIGRAMA = 2

# Incerteza (bootstrap por parágrafos/itens)
N_BOOTSTRAP = 2000
NIVEL_CONFIANCA = 0.95
SEMENTE = 2026

## 8. Ambiente, estilo e funções

Restrição de pasta, pasta de saída versionada, padrão gráfico e funções auxiliares.

In [ ]:
# ---------------------------------------------------------------------------
# Restrição de ambiente: o notebook só roda de dentro da pasta "Declarações IA".
# ---------------------------------------------------------------------------
PASTA = Path.cwd()
if unicodedata.normalize("NFC", PASTA.name) != "Declarações IA":
    raise RuntimeError(
        "Este notebook deve ser executado de dentro da pasta 'Declarações IA' "
        f"(restrição de ambiente da skill). Pasta atual: {PASTA}"
    )

# ---------------------------------------------------------------------------
# Pasta de saída versionada: resultados anteriores nunca são sobrescritos.
# ---------------------------------------------------------------------------
DIR_SAIDA = PASTA / "resultados" / NOME_SAIDA / VERSAO_SAIDA
if SALVAR_SAIDAS:
    if DIR_SAIDA.exists() and any(DIR_SAIDA.iterdir()) and not SOBRESCREVER:
        raise FileExistsError(
            f"A pasta de saída '{DIR_SAIDA.relative_to(PASTA)}' já contém arquivos. "
            "Para preservar a versão anterior, altere VERSAO_SAIDA (ex.: 'v2') na configuração. "
            "Use SOBRESCREVER = True somente se a substituição for intencional."
        )
    DIR_SAIDA.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Estilo das figuras (padrão acadêmico: fonte >= 9 pt, 300 dpi, PNG + PDF)
# ---------------------------------------------------------------------------
# Cores fixas por documento, iguais em todos os notebooks da pasta.
# O par laranja/azul foi verificado para leitores com daltonismo.
COR_DOCUMENTO = {"BRICS": "#eb6834", "OCDE": "#2a78d6"}
COR_TINTA = "#0b0b0b"     # texto principal
COR_TINTA_2 = "#52514e"   # texto secundário (rótulos, anotações)
COR_GRADE = "#e4e3df"     # grade e linhas de apoio
COR_LIGACAO = "#b9b8b2"   # segmentos de ligação (gráficos de halteres)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": COR_TINTA_2,
    "axes.linewidth": 0.6,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": COR_GRADE,
    "grid.linewidth": 0.6,
    "grid.linestyle": "-",
    "text.color": COR_TINTA,
    "axes.labelcolor": COR_TINTA,
    "xtick.color": COR_TINTA_2,
    "ytick.color": COR_TINTA_2,
    "legend.frameon": False,
    "pdf.fonttype": 42,
})

# ---------------------------------------------------------------------------
# Funções auxiliares de formatação e gravação
# ---------------------------------------------------------------------------
def num(x, casas=2):
    """Formata número no padrão brasileiro (vírgula decimal, ponto de milhar)."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "—"
    texto = f"{x:,.{casas}f}"
    return texto.replace(",", "§").replace(".", ",").replace("§", ".")


# Rótulos de eixo com vírgula decimal.
FMT_BR = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))


def exibir(df, casas=2):
    """Exibe uma tabela com vírgula decimal, sem alterar os valores."""
    display(df.style.format(precision=casas, decimal=",", thousands=".", na_rep="—"))


def salvar_figura(fig, nome):
    """Grava a figura em PNG (300 dpi) e PDF na pasta de saída versionada."""
    if SALVAR_SAIDAS:
        for ext in ("png", "pdf"):
            fig.savefig(DIR_SAIDA / f"{nome}.{ext}")


def salvar_tabela(df, nome, index=True):
    """Grava a tabela completa em CSV (UTF-8 com BOM, ';', vírgula decimal)."""
    if SALVAR_SAIDAS:
        df.to_csv(DIR_SAIDA / f"{nome}.csv", sep=";", decimal=",", encoding="utf-8-sig", index=index)


def semente_documento(sigla):
    """Semente do bootstrap própria de cada documento, derivada de SEMENTE e da sigla.
    Garante (a) os mesmos intervalos no notebook individual e no comparativo e
    (b) sequências aleatórias independentes entre documentos diferentes."""
    return np.random.SeedSequence([SEMENTE, zlib.crc32(sigla.encode("utf-8"))])


print(f"Pasta de trabalho: {PASTA}")
print(f"Saídas: {DIR_SAIDA.relative_to(PASTA) if SALVAR_SAIDAS else 'gravação desativada'}")

COR = COR_DOCUMENTO.get(SIGLA, "#2a78d6")

In [ ]:
# ===========================================================================
# PIPELINE DE PRÉ-PROCESSAMENTO E MEDIDAS
# (idêntico em todos os notebooks de análise da pasta "Declarações IA")
# ===========================================================================

# Token (palavra) = sequência de letras, admitindo hífen interno (multi-stakeholder)
# ou sigla com pontos (U.S.). Números, pontuação e símbolos NÃO são tokens.
RE_PALAVRA = re.compile(r"(?:[^\W\d_]\.){2,}|[^\W\d_]+(?:[-‐‑][^\W\d_]+)*")
HIFENS = {"-", "‐", "‑"}


def normalizar_hifen(texto):
    """Uniformiza variantes tipográficas do hífen (apenas nas chaves de contagem;
    o texto do JSON não é alterado)."""
    return texto.replace("‐", "-").replace("‑", "-")


def carregar_extracao(caminho):
    """Lê o JSON produzido pela skill 01 e devolve SOMENTE o título e as unidades
    (id, texto integral). Todo o conteúdo periférico é descartado aqui.
    Interrompe a execução se o arquivo não passar nas verificações de integridade."""
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo '{caminho.name}' não encontrado na pasta. "
            "Execute antes a skill 01 (skill_extração.md) para gerar o JSON da declaração."
        )
    bruto = caminho.read_bytes()
    dados = json.loads(bruto.decode("utf-8"))

    problemas = [f"chave '{c}' ausente" for c in ("document", "extraction", "sections") if c not in dados]
    if problemas:
        raise ValueError("Falha na verificação do JSON:\n- " + "\n- ".join(problemas))

    extracao = dados["extraction"]
    if extracao.get("skill") != "skill_extração":
        problemas.append("o JSON não registra ter sido produzido pela skill 01 (extraction.skill)")
    secoes = dados["sections"]
    if [s.get("id") for s in secoes] != list(range(1, len(secoes) + 1)):
        problemas.append("os ids das unidades não são sequenciais a partir de 1")
    vazias = [s.get("id") for s in secoes if not isinstance(s.get("text"), str) or not s["text"].strip()]
    if vazias:
        problemas.append(f"unidades com texto vazio: {vazias}")
    if extracao.get("units_extracted") is not None and extracao["units_extracted"] != len(secoes):
        problemas.append(
            f"extraction.units_extracted = {extracao['units_extracted']}, mas há {len(secoes)} unidades"
        )
    if not dados["document"].get("title"):
        problemas.append("título (document.title) ausente")
    if problemas:
        raise ValueError("Falha na verificação do JSON:\n- " + "\n- ".join(problemas))

    return {
        "arquivo": caminho.name,
        "sha256": hashlib.sha256(bruto).hexdigest(),
        "titulo": dados["document"]["title"],
        "idioma": dados["document"].get("language"),
        "pendencias_revisao": extracao.get("items_flagged_for_review"),
        "unidades": [(s["id"], s["text"]) for s in secoes],   # SOMENTE id (posição) e texto
    }


def montar_stopwords(nlp):
    """Lista de stopwords do spaCy, ajustada pelas listas editáveis da configuração."""
    base = {w.lower() for w in nlp.Defaults.stop_words}
    return (base | {w.lower() for w in STOPWORDS_ADICIONAIS}) - {w.lower() for w in STOPWORDS_MANTIDAS}


def unir_compostos_hifenizados(doc):
    """Reúne em um único token as palavras compostas com hífen (ex.: multi-stakeholder,
    human-centric, state-of-the-art), que o spaCy separa em vários tokens.
    Só une quando não há espaço em volta do hífen."""
    trechos, i, n = [], 0, len(doc)
    while i < n:
        j = i
        while (j + 2 < n and doc[j].is_alpha and doc[j].whitespace_ == ""
               and doc[j + 1].text in HIFENS and doc[j + 1].whitespace_ == ""
               and doc[j + 2].is_alpha):
            j += 2
        if j > i:
            trechos.append(doc[i:j + 1])
            i = j + 1
        else:
            i += 1
    with doc.retokenize() as rt:
        for trecho in trechos:
            ultimo = trecho[-1]
            # Lema do composto = forma escrita; se o último elemento estiver no plural,
            # usa-se o lema dele (multi-stakeholders -> multi-stakeholder).
            if ultimo.tag_ in {"NNS", "NNPS"}:
                lema = trecho.text[: len(trecho.text) - len(ultimo.text)] + ultimo.lemma_
            else:
                lema = trecho.text
            rt.merge(trecho, attrs={"LEMMA": lema, "POS": ultimo.pos_, "TAG": ultimo.tag_})
    return doc


def processar_documento(unidades, nlp, stopwords):
    """Aplica o pré-processamento a cada unidade (parágrafo/item) e devolve uma tabela
    com uma linha por token (palavra), na ordem original do documento."""
    linhas, posicao = [], 0
    for (uid, _), doc in zip(unidades, nlp.pipe([texto for _, texto in unidades])):
        doc = unir_compostos_hifenizados(doc)
        for k, frase in enumerate(doc.sents):
            for tok in frase:
                if not RE_PALAVRA.fullmatch(tok.text):
                    continue
                lema = normalizar_hifen(tok.lemma_.lower())
                lema = CORRECOES_DE_LEMA.get(lema, lema)
                linhas.append({
                    "unidade": uid,
                    "frase": f"{uid}.{k}",
                    "i": tok.i,                      # índice no parágrafo (detecta adjacência)
                    "posicao": posicao,              # posição no documento inteiro
                    "forma": normalizar_hifen(tok.text.lower()),
                    "lema": lema,
                    "classe": tok.pos_,
                    "etiqueta": tok.tag_,
                })
                posicao += 1
    tokens = pd.DataFrame(linhas)
    tokens["conteudo"] = (
        tokens["classe"].isin(CLASSES_DE_CONTEUDO)
        & ~tokens["lema"].isin(stopwords)
        & (tokens["lema"].str.len() >= TAMANHO_MINIMO_LEMA)
        & tokens["lema"].map(lambda lema: bool(RE_PALAVRA.fullmatch(lema)))
    )
    return tokens


def perfil_documento(tokens, n_unidades, rotulo):
    """Indicadores descritivos de extensão e vocabulário."""
    conteudo = tokens[tokens["conteudo"]]
    tam = tokens.groupby("unidade").size()
    freq_cont = conteudo["lema"].value_counts()
    return pd.Series({
        "Unidades (parágrafos/itens)": n_unidades,
        "Unidades sem nenhum token": n_unidades - tokens["unidade"].nunique(),
        "Tokens (palavras)": len(tokens),
        "Formas distintas": tokens["forma"].nunique(),
        "Tokens de conteúdo": len(conteudo),
        "Tokens de conteúdo (% dos tokens)": len(conteudo) / len(tokens) * 100,
        "Lemas de conteúdo distintos": conteudo["lema"].nunique(),
        "Lemas de conteúdo com uma única ocorrência": int((freq_cont == 1).sum()),
        "Tokens por unidade: média": tam.mean(),
        "Tokens por unidade: mediana": tam.median(),
        "Tokens por unidade: mínimo": tam.min(),
        "Tokens por unidade: máximo": tam.max(),
    }, name=rotulo)


def tabela_frequencias(tokens):
    """Frequência, taxa por 1.000 tokens, alcance e dispersão (DP normalizado) de
    TODOS os lemas de conteúdo. Ordenação: frequência decrescente; empate -> ordem alfabética."""
    n_tokens = len(tokens)
    unidades = sorted(tokens["unidade"].unique())
    proporcao_unidade = (tokens.groupby("unidade").size().reindex(unidades) / n_tokens).to_numpy()
    conteudo = tokens[tokens["conteudo"]]
    cruz = pd.crosstab(conteudo["lema"], conteudo["unidade"]).reindex(columns=unidades, fill_value=0)
    freq = cruz.sum(axis=1)
    # DP de Gries (2008), normalizado por Lijffijt & Gries (2012):
    # 0 = distribuição proporcional ao tamanho das unidades; 1 = concentração máxima.
    v = cruz.div(freq, axis=0).to_numpy()
    dp = 0.5 * np.abs(v - proporcao_unidade).sum(axis=1)
    dp_norm = dp / (1 - proporcao_unidade.min()) if len(unidades) > 1 else np.full(len(freq), np.nan)
    formas = conteudo.groupby("lema")["forma"].agg(
        lambda s: "; ".join(f"{forma} ({n})" for forma, n in s.value_counts().items())
    )
    tab = pd.DataFrame({
        "frequencia": freq,
        "por_1000_tokens": freq / n_tokens * 1000,
        "unidades_com_ocorrencia": (cruz > 0).sum(axis=1),
        "pct_unidades": (cruz > 0).sum(axis=1) / len(unidades) * 100,
        "DP_norm": dp_norm,
        "formas_observadas": formas,
    })
    tab.index.name = "lema"
    tab = tab.reset_index().sort_values(["frequencia", "lema"], ascending=[False, True])
    return tab.set_index("lema")


def relatar_empates(tab, coluna, n, rotulo):
    """Informa termos empatados no ponto de corte de um recorte 'top N'."""
    if len(tab) <= n:
        return
    corte = tab[coluna].iloc[n - 1]
    fora = tab.iloc[n:]
    fora = fora[fora[coluna] == corte]
    if len(fora):
        lista = ", ".join(fora.index[:20]) + (" …" if len(fora) > 20 else "")
        print(
            f"Empate no ponto de corte: o {n}º {rotulo} tem valor {num(corte, 2)}; "
            f"outros {len(fora)} com o mesmo valor ficaram fora do recorte "
            f"(desempate por ordem alfabética): {lista}"
        )


def tabela_bigramas(tokens):
    """Pares de tokens de conteúdo ADJACENTES no texto original (mesma frase, sem
    pontuação entre eles). Não se formam pares artificiais pela remoção de stopwords."""
    a = tokens.iloc[:-1].reset_index(drop=True)
    b = tokens.iloc[1:].reset_index(drop=True)
    ok = (
        (a["frase"].to_numpy() == b["frase"].to_numpy())
        & (b["i"].to_numpy() - a["i"].to_numpy() == 1)
        & a["conteudo"].to_numpy() & b["conteudo"].to_numpy()
    )
    pares = pd.DataFrame({
        "bigrama": (a["lema"] + " " + b["lema"])[ok],
        "forma": (a["forma"] + " " + b["forma"])[ok],
        "unidade": a["unidade"][ok],
    })
    if pares.empty:
        return pd.DataFrame(columns=["frequencia", "por_1000_tokens", "unidades_com_ocorrencia",
                                     "forma_principal", "formas_observadas"]).rename_axis("bigrama")
    grupos = pares.groupby("bigrama")
    tab = pd.DataFrame({
        "frequencia": grupos.size(),
        "por_1000_tokens": grupos.size() / len(tokens) * 1000,
        "unidades_com_ocorrencia": grupos["unidade"].nunique(),
        "forma_principal": grupos["forma"].agg(lambda s: s.value_counts().index[0]),
        "formas_observadas": grupos["forma"].agg(
            lambda s: "; ".join(f"{f} ({n})" for f, n in s.value_counts().items())
        ),
    })
    tab = tab.reset_index().sort_values(["frequencia", "bigrama"], ascending=[False, True])
    return tab.set_index("bigrama")


def carregar_codebook(caminho):
    """Lê o codebook e verifica a exclusividade mútua das categorias.
    Devolve o codebook e a lista de entradas ordenada da expressão mais longa para a mais curta."""
    codebook = json.loads(caminho.read_text(encoding="utf-8"))
    entradas, vistos = [], {}
    for categoria, info in codebook["categorias"].items():
        for termo in info["termos"]:
            chave = tuple(normalizar_hifen(termo.lower()).split())
            if chave in vistos:
                raise ValueError(
                    f"Termo '{termo}' aparece em '{vistos[chave]}' e em '{categoria}'. "
                    "As categorias do codebook devem ser mutuamente exclusivas."
                )
            vistos[chave] = categoria
            entradas.append((chave, categoria, termo))
    entradas.sort(key=lambda e: -len(e[0]))
    sha = hashlib.sha256(caminho.read_bytes()).hexdigest()
    return codebook, entradas, sha


def aplicar_codebook(tokens, entradas):
    """Localiza os termos do codebook no texto. Cada palavra do termo é comparada com a
    forma OU com o lema do token. Expressões exigem palavras contíguas na mesma frase.
    Prevalece a expressão mais longa, e cada token é contado no máximo uma vez."""
    achados = []
    for _, frase in tokens.groupby("frase", sort=False):
        formas = frase["forma"].tolist()
        lemas = frase["lema"].tolist()
        idx = frase["i"].tolist()
        pos = frase["posicao"].tolist()
        uid = frase["unidade"].iloc[0]
        j, n = 0, len(formas)
        while j < n:
            for chave, categoria, termo in entradas:
                tam = len(chave)
                if j + tam > n:
                    continue
                if all(formas[j + k] == chave[k] or lemas[j + k] == chave[k] for k in range(tam)) and \
                   all(idx[j + k + 1] - idx[j + k] == 1 for k in range(tam - 1)):
                    achados.append({"unidade": uid, "posicao": pos[j], "categoria": categoria,
                                    "termo": termo, "forma": " ".join(formas[j:j + tam]), "n_tokens": tam})
                    j += tam
                    break
            else:
                j += 1
    return pd.DataFrame(achados, columns=["unidade", "posicao", "categoria", "termo", "forma", "n_tokens"])


def taxas_bootstrap(contagens, tamanhos, semente):
    """Bootstrap por unidades (parágrafos/itens): reamostra unidades com reposição e
    recalcula a taxa por 1.000 tokens. Respeita a dependência entre palavras de um
    mesmo parágrafo. contagens: matriz (rótulos x unidades). Devolve (rótulos x reamostras)."""
    rng = np.random.default_rng(semente)
    n_u = len(tamanhos)
    idx = rng.integers(0, n_u, size=(N_BOOTSTRAP, n_u))
    return contagens[:, idx].sum(axis=2) / tamanhos[idx].sum(axis=1) * 1000


def tabela_categorias(achados, tokens, categorias, semente):
    """Ocorrências por categoria, taxa por 1.000 tokens com IC por bootstrap,
    participação no total do codebook e alcance."""
    unidades = sorted(tokens["unidade"].unique())
    tamanhos = tokens.groupby("unidade").size().reindex(unidades).to_numpy()
    if achados.empty:
        matriz = pd.DataFrame(0, index=categorias, columns=unidades)
    else:
        matriz = pd.crosstab(achados["categoria"], achados["unidade"]).reindex(
            index=categorias, columns=unidades, fill_value=0)
    reamostras = taxas_bootstrap(matriz.to_numpy(), tamanhos, semente)
    alfa = (1 - NIVEL_CONFIANCA) / 2
    ocorr = matriz.sum(axis=1)
    tab = pd.DataFrame({
        "ocorrencias": ocorr,
        "por_1000_tokens": ocorr / len(tokens) * 1000,
        "ic_inferior": np.quantile(reamostras, alfa, axis=1),
        "ic_superior": np.quantile(reamostras, 1 - alfa, axis=1),
        "pct_do_codebook": ocorr / ocorr.sum() * 100 if ocorr.sum() else np.nan,
        "unidades_com_ocorrencia": (matriz > 0).sum(axis=1),
    })
    tab.index.name = "categoria"
    return tab, reamostras


def tabela_termos_codebook(achados, entradas, n_tokens):
    """Contribuição de cada termo do codebook, incluindo os termos sem ocorrência."""
    linhas = []
    for _, categoria, termo in sorted(entradas, key=lambda e: (list(CATEGORIAS).index(e[1]), e[2])):
        sub = achados[(achados["categoria"] == categoria) & (achados["termo"] == termo)]
        linhas.append({
            "categoria": categoria,
            "termo": termo,
            "ocorrencias": len(sub),
            "por_1000_tokens": len(sub) / n_tokens * 1000,
            "formas_observadas": "; ".join(f"{f} ({n})" for f, n in sub["forma"].value_counts().items()),
        })
    tab = pd.DataFrame(linhas)
    total_cat = tab.groupby("categoria")["ocorrencias"].transform("sum")
    tab["pct_da_categoria"] = np.where(total_cat > 0, tab["ocorrencias"] / total_cat.where(total_cat > 0, 1) * 100, np.nan)
    return tab


def robustez_termo_dominante(termos, n_tokens):
    """Teste de robustez: quanto cada categoria depende do seu termo mais frequente."""
    linhas = []
    for categoria in CATEGORIAS:
        sub = termos[termos["categoria"] == categoria].sort_values(
            ["ocorrencias", "termo"], ascending=[False, True])
        total = sub["ocorrencias"].sum()
        dominante = sub.iloc[0]
        linhas.append({
            "categoria": categoria,
            "termo_dominante": dominante["termo"] if total else "—",
            "ocorrencias_do_termo": dominante["ocorrencias"] if total else 0,
            "participacao_do_termo_pct": dominante["ocorrencias"] / total * 100 if total else np.nan,
            "por_1000_com_o_termo": total / n_tokens * 1000,
            "por_1000_sem_o_termo": (total - dominante["ocorrencias"]) / n_tokens * 1000,
        })
    tab = pd.DataFrame(linhas).set_index("categoria")
    tab["alerta"] = np.where(tab["participacao_do_termo_pct"] > 50,
                             "mais da metade da categoria vem de um único termo", "")
    return tab


MODAIS = ["must", "shall", "should", "will", "would", "may", "might", "can", "could"]


def tabela_modais(tokens):
    """Verbos modais (etiqueta gramatical MD), por lema, por 1.000 tokens.
    Ordem fixa; modais fora da lista padrão, se houver, entram ao final."""
    modais = tokens[tokens["etiqueta"] == "MD"]["lema"].value_counts()
    ordem = MODAIS + sorted(set(modais.index) - set(MODAIS))
    modais = modais.reindex(ordem, fill_value=0)
    return pd.DataFrame({"ocorrencias": modais, "por_1000_tokens": modais / len(tokens) * 1000}).rename_axis("modal")


def grafico_dispersao_posicional(ax, posicoes_por_linha, rotulos, inicios_unidades, cor):
    """Gráfico de dispersão posicional: cada traço é uma ocorrência, na posição
    relativa em que aparece no documento (0 = início; 1 = fim)."""
    if len(inicios_unidades) <= 120:
        for x in inicios_unidades:
            ax.axvline(x, color=COR_GRADE, lw=0.5, zorder=0)
    for y, posicoes in enumerate(posicoes_por_linha):
        ax.vlines(posicoes, y - 0.36, y + 0.36, color=cor, lw=0.9)
    ax.set_yticks(range(len(rotulos)), labels=rotulos)
    ax.set_ylim(len(rotulos) - 0.5, -0.5)
    ax.set_xlim(0, 1)
    ax.grid(False)
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel("Posição relativa no documento (0 = início; 1 = fim)")


def versoes_ambiente(nlp):
    return {
        "python": platform.python_version(),
        "spacy": spacy.__version__,
        "modelo_spacy": f"{MODELO_SPACY} {nlp.meta.get('version', '?')}",
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
    }

## 9. Carregamento e verificação do insumo

O JSON da skill 01 é lido em modo somente leitura. A execução é interrompida se: o arquivo não existir; não tiver sido produzido pela skill 01; os identificadores não forem sequenciais; houver unidade vazia; ou o número de unidades divergir do registrado na extração. Pendências de revisão humana registradas no relatório CSV da skill 01 são sinalizadas.

In [ ]:
documento = carregar_extracao(PASTA / ARQUIVO_JSON)
unidades = list(documento["unidades"])
if INCLUIR_TITULO_NA_CONTAGEM:
    unidades = [(0, documento["titulo"])] + unidades

if documento["idioma"] != IDIOMA_ESPERADO:
    raise ValueError(
        f"Idioma declarado no JSON: {documento['idioma']!r}; esperado: {IDIOMA_ESPERADO!r}. "
        "Ajuste IDIOMA_ESPERADO e MODELO_SPACY para o idioma do documento."
    )

print(f"Título: {documento['titulo']}")
print(f"Arquivo: {documento['arquivo']}")
print(f"SHA-256: {documento['sha256']}")
print(f"Idioma: {documento['idioma']}")
print(f"Unidades no JSON: {len(documento['unidades'])}")
print(f"Título incluído nas contagens: {'sim' if INCLUIR_TITULO_NA_CONTAGEM else 'não'}")
if documento["pendencias_revisao"]:
    print(
        f"\nATENÇÃO: o relatório CSV da skill 01 registra {documento['pendencias_revisao']} item(ns) "
        "marcado(s) para revisão humana. Os resultados abaixo dependem da decisão do pesquisador sobre esses itens."
    )

## 10. Pré-processamento e perfil do documento

In [ ]:
nlp = spacy.load(MODELO_SPACY)
STOPWORDS = montar_stopwords(nlp)
tokens = processar_documento(unidades, nlp, STOPWORDS)
N_TOKENS = len(tokens)
N_UNIDADES = tokens["unidade"].nunique()

perfil = perfil_documento(tokens, len(unidades), SIGLA).to_frame()
exibir(perfil)
salvar_tabela(perfil, "00_perfil_documento")
salvar_tabela(tokens, "00_tokens_processados", index=False)

print(f"Stopwords aplicadas: {len(STOPWORDS)} (lista do spaCy, {len(STOPWORDS_ADICIONAIS)} adicionada(s), "
      f"{len(STOPWORDS_MANTIDAS)} mantida(s))")
print(f"Correções de lema aplicadas: {CORRECOES_DE_LEMA}")

**Método e escolhas.** Cada unidade é processada com o modelo `en_core_web_sm` do spaCy. Palavras compostas com hífen (*multi-stakeholder*) são reunidas em um só token. **Token** é toda palavra formada por letras (com hífen interno ou sigla com pontos); números e pontuação não são tokens. O total de tokens é o **denominador** de todas as taxas por 1.000 tokens. **Tokens de conteúdo** são substantivos, nomes próprios, verbos, adjetivos e advérbios fora da lista de *stopwords* do spaCy, verificada pelo **lema**, para que todas as flexões de uma palavra recebam o mesmo tratamento. A tabela de tokens processados é exportada (`00_tokens_processados.csv`) e permite conferir, token a token, a forma, o lema, a classe gramatical e a decisão de inclusão.

**Como ler.** O perfil descreve a extensão do documento e de seu vocabulário. Ele serve de referência para dimensionar as frequências apresentadas a seguir.

**Fragilidades e ajustes possíveis.** O lematizador e o etiquetador do spaCy cometem erros, sobretudo em siglas e termos técnicos. Por isso, as tabelas trazem as formas observadas de cada lema. Correções conhecidas ficam explícitas em `CORRECOES_DE_LEMA`. A lista de *stopwords* do spaCy inclui palavras como *may*, *must* e *should*, que são excluídas da contagem de conteúdo, mas analisadas à parte na seção de verbos modais.

## 11. Extensão das unidades textuais

In [ ]:
tam_unidades = tokens.groupby("unidade").size().reindex([u for u, _ in unidades], fill_value=0)

fig, ax = plt.subplots(figsize=(7.5, 3.2))
x = np.arange(len(tam_unidades))
ax.bar(x, tam_unidades.to_numpy(), width=0.8, color=COR)
ax.axhline(tam_unidades.mean(), color=COR_TINTA_2, lw=0.8)
ax.text(len(x) - 0.5, tam_unidades.mean(), f"média = {num(tam_unidades.mean(), 1)}",
        ha="right", va="bottom", fontsize=9, color=COR_TINTA_2)
passo = max(1, int(np.ceil(len(x) / 20)))
ax.set_xticks(x[::passo], labels=[str(u) for u in tam_unidades.index[::passo]])
ax.set_xlim(-0.6, len(x) - 0.4)
ax.set_xlabel("Unidade (id do parágrafo ou item), na ordem do documento")
ax.set_ylabel("Tokens (palavras)")
ax.set_title(f"{SIGLA} · extensão das unidades textuais", loc="left")
ax.grid(axis="x", visible=False)
ax.yaxis.set_major_formatter(FMT_BR)
salvar_figura(fig, "01_extensao_unidades")
plt.show()

salvar_tabela(tam_unidades.rename("tokens").to_frame(), "01_extensao_unidades")
maior, menor = tam_unidades.idxmax(), tam_unidades.idxmin()
print(f"Síntese factual: {len(tam_unidades)} unidades; média de {num(tam_unidades.mean(), 1)} tokens "
      f"(desvio-padrão {num(tam_unidades.std(), 1)}). Maior unidade: id {maior} ({tam_unidades[maior]} tokens); "
      f"menor: id {menor} ({tam_unidades[menor]} tokens).")

**Método e escolhas.** Número de tokens (palavras) de cada unidade, na ordem original do documento. A linha horizontal marca a média. Nenhuma unidade é omitida.

**Como ler.** Cada barra é um parágrafo ou item. Barras altas indicam unidades extensas; a sequência mostra como o texto se organiza (por exemplo, um preâmbulo de parágrafos longos seguido de itens curtos).

**Interpretação.** *[Pendente. A redigir somente após a execução do notebook com o JSON definitivo da skill 01, exclusivamente a partir dos valores exibidos acima, separando observação de inferência (skill 02, seção 4.3).]*

**Fragilidades e ajustes possíveis.** A segmentação em unidades é a da skill 01, que segue a estrutura do original. Unidades muito curtas, como itens de lista, tornam instáveis as medidas por unidade (alcance e dispersão), o que deve ser lembrado na leitura das seções seguintes.

## 12. Lemas de conteúdo mais frequentes

In [ ]:
freq = tabela_frequencias(tokens)
salvar_tabela(freq, "02_frequencias_lemas_completa")
relatar_empates(freq, "frequencia", TOP_N_LEMAS, "lema")
top = freq.head(TOP_N_LEMAS)

fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(top) + 1.3))
y = np.arange(len(top))
ax.barh(y, top["frequencia"], height=0.7, color=COR)
ax.set_yticks(y, labels=top.index)
ax.invert_yaxis()
xmax = top["frequencia"].max()
for yi, (_, linha) in zip(y, top.iterrows()):
    ax.text(linha["frequencia"] + xmax * 0.015, yi,
            f"{num(linha['por_1000_tokens'])} por 1.000 · {int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
            va="center", fontsize=9, color=COR_TINTA_2)
ax.set_xlim(0, xmax * 1.55)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.xaxis.set_major_formatter(FMT_BR)
ax.set_xlabel("Frequência absoluta (ocorrências)")
ax.set_title(f"{SIGLA} · {len(top)} lemas de conteúdo mais frequentes", loc="left")
ax.grid(axis="y", visible=False)
salvar_figura(fig, "02_lemas_mais_frequentes")
plt.show()

exibir(top[["frequencia", "por_1000_tokens", "unidades_com_ocorrencia", "pct_unidades", "DP_norm", "formas_observadas"]])
n_conteudo = int(tokens["conteudo"].sum())
print(f"Síntese factual: os {len(top)} lemas acima somam {int(top['frequencia'].sum())} ocorrências, "
      f"{num(top['frequencia'].sum() / n_conteudo * 100, 1)}% dos {n_conteudo} tokens de conteúdo. "
      f"O documento tem {len(freq)} lemas de conteúdo distintos (tabela completa exportada).")

**Método e escolhas.** Frequência absoluta dos *N* lemas de conteúdo mais frequentes (*N* = `TOP_N_LEMAS`). O rótulo de cada barra traz a frequência por 1.000 tokens e o **alcance** (em quantas unidades o lema aparece). A tabela acrescenta o **DP normalizado** (dispersão) e as formas observadas. Ordenação: frequência decrescente; empates resolvidos por ordem alfabética e informados acima do gráfico quando afetam o corte. A tabela completa de todos os lemas é exportada (`02_frequencias_lemas_completa.csv`).

**Como ler.** Barras mais longas indicam lemas mais frequentes. Um lema frequente presente em muitas unidades é estruturante no texto; um lema frequente presente em poucas unidades é localizado. A distinção aparece no rótulo (alcance) e na coluna DP_norm da tabela (0 = espalhado; 1 = concentrado).

**Interpretação.** *[Pendente. A redigir somente após a execução do notebook com o JSON definitivo da skill 01, exclusivamente a partir dos valores exibidos acima, separando observação de inferência (skill 02, seção 4.3).]*

**Fragilidades e ajustes possíveis.** Frequência não equivale a importância política: um termo pode ser frequente por razões de estilo ou por repetição formular. Lemas genéricos, como o próprio *ai*, tendem a ocupar o topo em declarações sobre IA. Isso é um resultado, e não um erro, e por isso esses lemas não são removidos sem decisão do pesquisador. *AI* e *artificial intelligence* não são unificados, porque isso alteraria o texto. Se o pesquisador desejar, a unificação pode ser feita por regra explícita na configuração.

## 13. Distribuição dos lemas ao longo do documento

In [ ]:
lemas_disp = freq.head(TOP_N_DISPERSAO).index.tolist()
inicios = (tokens.groupby("unidade")["posicao"].min() / N_TOKENS).to_numpy()
posicoes = [tokens.loc[tokens["conteudo"] & (tokens["lema"] == lema), "posicao"].to_numpy() / N_TOKENS
            for lema in lemas_disp]
rotulos = [f"{lema}  (DP = {num(freq.loc[lema, 'DP_norm'])})" for lema in lemas_disp]

fig, ax = plt.subplots(figsize=(7.5, 0.33 * len(lemas_disp) + 1.3))
grafico_dispersao_posicional(ax, posicoes, rotulos, inicios, COR)
ax.set_title(f"{SIGLA} · posição das ocorrências dos {len(lemas_disp)} lemas mais frequentes", loc="left")
salvar_figura(fig, "03_dispersao_lemas")
plt.show()

exibir(freq.head(TOP_N_DISPERSAO)[["frequencia", "unidades_com_ocorrencia", "DP_norm"]].sort_values("DP_norm"))

**Método e escolhas.** Cada traço vertical é uma ocorrência do lema, posicionada pela sua posição relativa no documento (posição do token ÷ total de tokens). As linhas verticais claras marcam o início de cada unidade. O DP normalizado (Gries, 2008; Lijffijt & Gries, 2012) resume a distribuição: 0 indica ocorrências proporcionais ao tamanho das unidades; 1, concentração máxima.

**Como ler.** Linhas com traços espalhados indicam termos presentes em todo o texto; traços agrupados indicam termos concentrados em uma parte (por exemplo, em uma seção específica). A tabela ordena os lemas do mais espalhado ao mais concentrado.

**Interpretação.** *[Pendente. A redigir somente após a execução do notebook com o JSON definitivo da skill 01, exclusivamente a partir dos valores exibidos acima, separando observação de inferência (skill 02, seção 4.3).]*

**Fragilidades e ajustes possíveis.** Em documentos curtos ou com poucas unidades, o DP é sensível à segmentação: uma única unidade longa pode concentrar ocorrências por ser longa, e não por ênfase. O gráfico mostra posições, não intensidade. Para intensidade por tema, ver a seção de enfoque temático.

## 14. Expressões recorrentes (bigramas)

In [ ]:
bigramas = tabela_bigramas(tokens)
salvar_tabela(bigramas, "04_bigramas_completa")
elegiveis = bigramas[bigramas["frequencia"] >= FREQ_MINIMA_BIGRAMA]
relatar_empates(elegiveis, "frequencia", TOP_N_BIGRAMAS, "bigrama")
topb = elegiveis.head(TOP_N_BIGRAMAS)

if topb.empty:
    print(f"Nenhum bigrama atinge a frequência mínima de {FREQ_MINIMA_BIGRAMA}; gráfico não gerado.")
else:
    fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(topb) + 1.3))
    y = np.arange(len(topb))
    ax.barh(y, topb["frequencia"], height=0.7, color=COR)
    ax.set_yticks(y, labels=topb["forma_principal"])
    ax.invert_yaxis()
    xmax = topb["frequencia"].max()
    for yi, (_, linha) in zip(y, topb.iterrows()):
        ax.text(linha["frequencia"] + xmax * 0.015, yi,
                f"{int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
                va="center", fontsize=9, color=COR_TINTA_2)
    ax.set_xlim(0, xmax * 1.25)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel("Frequência absoluta (ocorrências)")
    ax.set_title(f"{SIGLA} · {len(topb)} expressões de duas palavras mais frequentes", loc="left")
    ax.grid(axis="y", visible=False)
    salvar_figura(fig, "04_bigramas")
    plt.show()
    exibir(topb)
print(f"Síntese factual: {len(bigramas)} bigramas distintos; {len(elegiveis)} com frequência >= {FREQ_MINIMA_BIGRAMA}.")

**Método e escolhas.** Pares de tokens de conteúdo **adjacentes no texto original**, na mesma frase e sem pontuação entre eles. Pares agrupados pelo lema (ex.: *developing country* reúne *developing countries*). O rótulo mostra a forma escrita mais frequente. Só entram no gráfico os bigramas com frequência mínima `FREQ_MINIMA_BIGRAMA`. Como a adjacência é verificada no texto original, a remoção de *stopwords* não cria pares artificiais (em *rights of persons*, não se forma *rights persons*).

**Como ler.** As barras mostram as expressões que o texto mais repete, frequentemente nomes de conceitos (*artificial intelligence*, *human rights*). O rótulo indica em quantas unidades a expressão aparece.

**Interpretação.** *[Pendente. A redigir somente após a execução do notebook com o JSON definitivo da skill 01, exclusivamente a partir dos valores exibidos acima, separando observação de inferência (skill 02, seção 4.3).]*

**Fragilidades e ajustes possíveis.** Expressões com três ou mais palavras aparecem fragmentadas em bigramas sobrepostos. Expressões interrompidas por *stopwords* (*rule of law*) não são captadas como bigramas, mas o codebook (seção seguinte) pode reconhecê-las. Em textos curtos, poucas expressões atingem a frequência mínima.

## 15. Enfoque temático (codebook de categorias)

O codebook é o mesmo para todas as declarações da pasta, para que os documentos sejam medidos com o mesmo instrumento. Ele é exibido por completo abaixo.

In [ ]:
codebook, ENTRADAS, SHA_CODEBOOK = carregar_codebook(PASTA / ARQUIVO_CODEBOOK)
CATEGORIAS = list(codebook["categorias"])
print(f"Codebook: {codebook.get('nome')} · versão {codebook.get('versao')}")
print(f"Status: {codebook.get('status')}")
print(f"SHA-256: {SHA_CODEBOOK}\n")
for regra in codebook.get("regras_de_correspondencia", []):
    print(f"- {regra}")
exibir(pd.DataFrame(
    [{"categoria": cat, "descrição": info["descricao"], "nº de termos": len(info["termos"]),
      "termos": ", ".join(info["termos"])} for cat, info in codebook["categorias"].items()]
).set_index("categoria"))

In [ ]:
achados = aplicar_codebook(tokens, ENTRADAS)
categorias, reamostras = tabela_categorias(achados, tokens, CATEGORIAS, semente_documento(SIGLA))
cobertura = achados["n_tokens"].sum() / N_TOKENS * 100
salvar_tabela(categorias, "05_categorias")
salvar_tabela(achados, "05_ocorrencias_codebook", index=False)

fig, ax = plt.subplots(figsize=(7.5, 0.5 * len(categorias) + 1.4))
y = np.arange(len(categorias))
valores = categorias["por_1000_tokens"].to_numpy()
erros = np.vstack([np.clip(valores - categorias["ic_inferior"], 0, None),
                   np.clip(categorias["ic_superior"] - valores, 0, None)])
ax.barh(y, valores, height=0.6, color=COR)
ax.errorbar(valores, y, xerr=erros, fmt="none", ecolor=COR_TINTA, elinewidth=1, capsize=3)
for yi, (_, linha) in zip(y, categorias.iterrows()):
    ax.text(linha["ic_superior"] + categorias["ic_superior"].max() * 0.02, yi,
            f"{num(linha['por_1000_tokens'])} ({num(linha['pct_do_codebook'], 1)}% do codebook)",
            va="center", fontsize=9, color=COR_TINTA_2)
ax.set_yticks(y, labels=categorias.index)
ax.invert_yaxis()
ax.set_xlim(0, categorias["ic_superior"].max() * 1.5)
ax.xaxis.set_major_formatter(FMT_BR)
ax.set_xlabel(f"Ocorrências por 1.000 tokens (barras de erro: IC {num(NIVEL_CONFIANCA * 100, 0)}% por bootstrap)")
ax.set_title(f"{SIGLA} · enfoque temático segundo o codebook", loc="left")
ax.grid(axis="y", visible=False)
salvar_figura(fig, "05_categorias")
plt.show()

exibir(categorias)
print(f"Síntese factual: o codebook capta {len(achados)} ocorrências, que cobrem {num(cobertura, 1)}% "
      f"dos {N_TOKENS} tokens do documento. Ordem das categorias: a do codebook (fixa entre documentos).")

**Método e escolhas.** Contagem das ocorrências dos termos de cada categoria do codebook, expressa em **ocorrências por 1.000 tokens** e em **percentual do total captado pelo codebook**. As barras de erro são intervalos de confiança por **bootstrap por unidades**: os parágrafos/itens são reamostrados com reposição (`N_BOOTSTRAP` vezes, semente fixa derivada de `SEMENTE` e da sigla), o que respeita a dependência entre palavras de um mesmo parágrafo. A ordem das categorias é a do codebook, igual em todos os documentos.

**Como ler.** Barras mais longas indicam categorias com maior presença relativa no texto. Barras de erro longas indicam que o valor depende de poucas unidades. A soma dos percentuais do codebook é 100% e mostra a **proporção** entre os enfoques, mas apenas dentro do vocabulário captado pelo codebook.

**Interpretação.** *[Pendente. A redigir somente após a execução do notebook com o JSON definitivo da skill 01, exclusivamente a partir dos valores exibidos acima, separando observação de inferência (skill 02, seção 4.3).]*

**Fragilidades e ajustes possíveis.** O codebook é uma **proposta inicial** e deve ser validado pelo pesquisador. Os resultados dependem diretamente da lista de termos. A **cobertura** informada acima mostra quanto do texto o codebook capta. Termos polissêmicos (*security*, *development*, *fair*) podem ser contados em sentidos alheios à categoria. As seções seguintes trazem a contribuição de cada termo e o teste de robustez, para identificar esses casos.

In [ ]:
termos_cb = tabela_termos_codebook(achados, ENTRADAS, N_TOKENS)
salvar_tabela(termos_cb, "05_termos_codebook", index=False)
print("Contribuição dos termos com ao menos uma ocorrência:")
exibir(termos_cb[termos_cb["ocorrencias"] > 0].set_index(["categoria", "termo"]))
zeros = termos_cb[termos_cb["ocorrencias"] == 0].groupby("categoria")["termo"].agg(", ".join)
print("\nTermos do codebook SEM ocorrência no documento:")
for categoria in CATEGORIAS:
    print(f"- {categoria}: {zeros.get(categoria, '(nenhum)')}")

In [ ]:
robustez = robustez_termo_dominante(termos_cb, N_TOKENS)
salvar_tabela(robustez, "05_robustez_categorias")
exibir(robustez)

**Teste de robustez: método e leitura.** Para cada categoria, identifica-se o termo mais frequente e recalcula-se a taxa da categoria **sem** ele. Se a participação desse termo passar de 50%, a categoria fica sinalizada: o resultado dela depende essencialmente de uma palavra. Nesse caso, a leitura deve se apoiar no termo, e não na categoria.

*Interpretação: [pendente — a redigir após a execução].*

## 16. Localização das categorias ao longo do documento

In [ ]:
posicoes_cat = [achados.loc[achados["categoria"] == cat, "posicao"].to_numpy() / N_TOKENS for cat in CATEGORIAS]
rotulos_cat = [f"{cat}  (n = {len(p)})" for cat, p in zip(CATEGORIAS, posicoes_cat)]

fig, ax = plt.subplots(figsize=(7.5, 0.45 * len(CATEGORIAS) + 1.3))
grafico_dispersao_posicional(ax, posicoes_cat, rotulos_cat, inicios, COR)
ax.set_title(f"{SIGLA} · posição das ocorrências de cada categoria", loc="left")
salvar_figura(fig, "06_dispersao_categorias")
plt.show()

**Método e escolhas.** Cada traço é uma ocorrência de um termo do codebook, posicionada pela sua posição relativa no documento. As linhas verticais claras marcam o início de cada unidade.

**Como ler.** Mostra **onde** cada enfoque aparece: espalhado pelo documento ou concentrado em uma parte. A sequência das concentrações indica como o texto encadeia os temas.

**Interpretação.** *[Pendente. A redigir somente após a execução do notebook com o JSON definitivo da skill 01, exclusivamente a partir dos valores exibidos acima, separando observação de inferência (skill 02, seção 4.3).]*

**Fragilidades e ajustes possíveis.** As mesmas ressalvas do codebook se aplicam. O gráfico não pondera pela extensão das unidades: trechos longos naturalmente acumulam mais traços.

## 17. Ênfase normativa: verbos modais

In [ ]:
modais = tabela_modais(tokens)
salvar_tabela(modais, "07_modais")

fig, ax = plt.subplots(figsize=(7.5, 0.32 * len(modais) + 1.3))
y = np.arange(len(modais))
ax.barh(y, modais["por_1000_tokens"], height=0.65, color=COR)
xmax = max(modais["por_1000_tokens"].max(), 1e-9)
for yi, (_, linha) in zip(y, modais.iterrows()):
    ax.text(linha["por_1000_tokens"] + xmax * 0.015, yi,
            f"{num(linha['por_1000_tokens'])} (n = {int(linha['ocorrencias'])})",
            va="center", fontsize=9, color=COR_TINTA_2)
ax.set_yticks(y, labels=modais.index)
ax.invert_yaxis()
ax.set_xlim(0, xmax * 1.35)
ax.xaxis.set_major_formatter(FMT_BR)
ax.set_xlabel("Ocorrências por 1.000 tokens")
ax.set_title(f"{SIGLA} · verbos modais", loc="left")
ax.grid(axis="y", visible=False)
salvar_figura(fig, "07_modais")
plt.show()
print(f"Síntese factual: {int(modais['ocorrencias'].sum())} verbos modais no documento "
      f"({num(modais['ocorrencias'].sum() / N_TOKENS * 1000)} por 1.000 tokens).")

**Método e escolhas.** Contagem dos verbos modais identificados pela etiqueta gramatical `MD` do spaCy, por lema, em ocorrências por 1.000 tokens. A ordem é fixa, das formas de obrigação (*must*, *shall*, *should*) às de possibilidade (*may*, *might*, *can*, *could*). Modais fora da lista padrão, se houver, entram ao final.

**Como ler.** Modais de obrigação (*shall*, *must*) indicam linguagem prescritiva; *should* indica recomendação; *will* pode indicar compromisso; *may*/*can*, possibilidade ou permissão. O perfil dos modais caracteriza o grau de compromisso normativo do texto.

**Interpretação.** *[Pendente. A redigir somente após a execução do notebook com o JSON definitivo da skill 01, exclusivamente a partir dos valores exibidos acima, separando observação de inferência (skill 02, seção 4.3).]*

**Fragilidades e ajustes possíveis.** *Will* é ambíguo: pode expressar compromisso (*we will promote*) ou simples futuro (*AI will transform*). Separar esses usos exigiria leitura qualitativa das ocorrências, e a tabela de tokens exportada permite fazê-la. Compromissos expressos sem modal (*we commit to*, *we affirm*) não são captados por esta medida.

## 18. Registro de execução

In [ ]:
registro = {
    "notebook": f"Análise_{SIGLA}.ipynb",
    "skill": "skill_análise_individual (etapa 02)",
    "data_execucao": datetime.now().isoformat(timespec="seconds"),
    "insumo": {"arquivo": documento["arquivo"], "sha256": documento["sha256"], "titulo": documento["titulo"],
               "pendencias_revisao_skill01": documento["pendencias_revisao"]},
    "codebook": {"arquivo": ARQUIVO_CODEBOOK, "versao": codebook.get("versao"), "status": codebook.get("status"),
                 "sha256": SHA_CODEBOOK},
    "parametros": {
        "INCLUIR_TITULO_NA_CONTAGEM": INCLUIR_TITULO_NA_CONTAGEM, "MODELO_SPACY": MODELO_SPACY,
        "CLASSES_DE_CONTEUDO": sorted(CLASSES_DE_CONTEUDO), "TAMANHO_MINIMO_LEMA": TAMANHO_MINIMO_LEMA,
        "STOPWORDS_ADICIONAIS": STOPWORDS_ADICIONAIS, "STOPWORDS_MANTIDAS": STOPWORDS_MANTIDAS,
        "CORRECOES_DE_LEMA": CORRECOES_DE_LEMA, "TOP_N_LEMAS": TOP_N_LEMAS, "TOP_N_DISPERSAO": TOP_N_DISPERSAO,
        "TOP_N_BIGRAMAS": TOP_N_BIGRAMAS, "FREQ_MINIMA_BIGRAMA": FREQ_MINIMA_BIGRAMA,
        "N_BOOTSTRAP": N_BOOTSTRAP, "NIVEL_CONFIANCA": NIVEL_CONFIANCA, "SEMENTE": SEMENTE,
    },
    "resultados_principais": {"unidades": len(unidades), "tokens": N_TOKENS,
                              "cobertura_codebook_pct": round(float(cobertura), 4)},
    "versoes": versoes_ambiente(nlp),
}
if SALVAR_SAIDAS:
    registro["saidas"] = sorted(p.name for p in DIR_SAIDA.iterdir()) + ["registro_execucao.json"]
    (DIR_SAIDA / "registro_execucao.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(registro, ensure_ascii=False, indent=2))

## 19. Síntese da análise

*[Pendente. A redigir após a execução, a partir das interpretações de cada seção: principais termos, enfoques predominantes, proporções entre categorias, ênfase normativa e fragilidades que limitam as conclusões. A síntese descreve apenas este documento; comparações com outras declarações são feitas no notebook comparativo (skill 03).]*